# Memory & Performance Optimization

Companion notebook for the [Memory & Performance lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/03-memory-and-performance).

We quantify the three big levers from the lesson: **memory coalescing** (model bandwidth efficiency
vs. access stride), **tiling** (count global-memory reads for naive vs. tiled matmul), and the
**roofline model** (plot the compute/memory ceilings and place real ops on them). Pure NumPy +
Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Coalescing efficiency vs. stride

A warp of 32 threads reads addresses `t * stride`. Memory arrives in fixed transactions (a cache
line of `SEG` cells). Efficiency = bytes the threads actually use ÷ bytes the hardware fetched.
Stride 1 is fully coalesced (~100%); larger strides scatter the reads across more transactions.

In [ ]:
def coalescing_efficiency(stride, warp=32, seg=32):
    addresses = np.arange(warp) * stride
    transactions = len(set(addresses // seg))      # distinct cache lines touched
    bytes_fetched = transactions * seg
    return warp / bytes_fetched

strides = [1, 2, 4, 8, 16, 32]
eff = [coalescing_efficiency(s) for s in strides]

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar([str(s) for s in strides], [e * 100 for e in eff], color='#6366f1')
ax.set_xlabel('access stride'); ax.set_ylabel('bandwidth efficiency (%)')
ax.set_title('Coalescing: stride 1 uses all fetched bytes; striding wastes them')
ax.grid(True, alpha=0.3, axis='y')
for b, e in zip(bars, eff):
    ax.text(b.get_x() + b.get_width()/2, e*100 + 1, f'{e*100:.0f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

for s, e in zip(strides, eff):
    print(f"stride {s:2d}: efficiency {e*100:5.1f}%")

## 2 — Tiling cuts global-memory traffic

Naive matmul re-reads each input element from slow global memory once per output it contributes to.
Tiling loads a `T x T` tile into shared memory once and reuses it across the tile, cutting global
reads by ~`T`. We count global-memory reads for an `N x N` matmul both ways.

In [ ]:
def global_reads_naive(N):
    # each of N*N outputs reads N from A and N from B
    return N * N * (2 * N)

def global_reads_tiled(N, T):
    # tiles of size T; each (T x T) output tile loads (N/T) pairs of T x T input tiles once
    tiles = (N // T) ** 2
    loads_per_tile = (N // T) * (2 * T * T)
    return tiles * loads_per_tile

N = 1024
for T in [8, 16, 32]:
    naive = global_reads_naive(N)
    tiled = global_reads_tiled(N, T)
    print(f"N={N}, tile T={T:2d}:  naive={naive/1e9:.2f}G reads  tiled={tiled/1e9:.2f}G  -> {naive/tiled:.1f}x less traffic")

## 3 — The roofline model

Achievable performance is capped by $\min(P_{\max},\, I \times BW)$ where $I$ is arithmetic intensity
(FLOP/byte). Low-$I$ ops are **memory-bound** (left of the ridge); high-$I$ ops are **compute-bound**
(right). We use representative numbers ($P_{\max}$ = 20 TFLOP/s, $BW$ = 1 TB/s).

In [ ]:
P_max = 20e12     # peak FLOP/s
BW = 1e12         # bytes/s
ridge = P_max / BW
print(f"ridge point I* = P_max / BW = {ridge:.0f} FLOP/byte")

I = np.logspace(-1, 3, 200)
perf = np.minimum(P_max, I * BW)

ops = {
    'vector add (~0.08)': 0.08,
    'activation (~0.5)':  0.5,
    'small matmul (~8)':  8,
    'large matmul (~200)': 200,
}
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(I, perf / 1e12, color='#818cf8')
ax.axvline(ridge, ls='--', color='#555', label=f'ridge I*={ridge:.0f}')
for name, intensity in ops.items():
    p = min(P_max, intensity * BW)
    bound = 'memory' if intensity < ridge else 'compute'
    c = '#fb7185' if bound == 'memory' else '#2dd4bf'
    ax.plot(intensity, p / 1e12, 'o', color=c, ms=9)
    ax.annotate(f'{name}\n[{bound}-bound]', (intensity, p/1e12),
                textcoords='offset points', xytext=(6, -18), fontsize=7, color=c)
ax.set_xlabel('arithmetic intensity I (FLOP/byte)'); ax.set_ylabel('performance (TFLOP/s)')
ax.set_title('Roofline: memory-bound (left) vs compute-bound (right)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Exercise.** Implement `roofline_perf(I, P_max, BW)` returning achievable performance, and
`is_memory_bound(I, P_max, BW)` returning `True` when the op is limited by bandwidth (left of the
ridge point $I^* = P_{\max}/BW$).

In [ ]:
def roofline_perf(I, P_max, BW):
    # TODO(you): achievable performance is the lower of the compute and memory ceilings
    return ...

def is_memory_bound(I, P_max, BW):
    # TODO(you): True if intensity is below the ridge point
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert roofline_perf(0.5, 20e12, 1e12) == 0.5e12        # memory-bound: I*BW
assert roofline_perf(200, 20e12, 1e12) == 20e12         # compute-bound: P_max
assert is_memory_bound(0.5, 20e12, 1e12) is True
assert is_memory_bound(200, 20e12, 1e12) is False
assert coalescing_efficiency(1) == 1.0                  # stride 1 fully coalesced
assert coalescing_efficiency(2) == 0.5                  # stride 2 wastes half
print("\u2713 roofline + coalescing checks pass")

<details>
<summary>Solution</summary>

```python
def roofline_perf(I, P_max, BW):
    return min(P_max, I * BW)

def is_memory_bound(I, P_max, BW):
    return I < P_max / BW
```

The ridge point $I^* = P_{\max}/BW$ is where the two ceilings meet. Below it, raising FLOPs does
nothing — you must raise intensity (fuse, tile) or bandwidth (coalesce). Above it, you're limited by
raw arithmetic throughput, where tensor cores and low precision help.

</details>